<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/Tabular_Res_net_MLP_final_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Reproducibility & Data Load
tf.keras.utils.set_random_seed(42)
data = pd.read_csv('/content/GTPP-DATASET.csv')

X = data[['AT', 'AP', 'DF', 'CH', 'GP', 'CPR', 'CPD', 'TTXM']].values
y = data['EP'].values

# 80:20 split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 2. Thermodynamic Pairwise Interactions (8 features -> 44 physical interaction terms)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_raw)
X_test_poly = poly.transform(X_test_raw)

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_poly)
X_test = scaler.transform(X_test_poly)

# 3. Compact Tabular ResNet Architecture
def build_tabular_resnet(input_dim):
    inputs = tf.keras.layers.Input(shape=(input_dim,))

    # Initial projection
    x = tf.keras.layers.Dense(128, activation='swish')(inputs)
    x = tf.keras.layers.LayerNormalization()(x)

    # Residual Block 1
    shortcut = x
    r = tf.keras.layers.Dense(128, activation='swish')(x)
    r = tf.keras.layers.Dropout(0.1)(r)
    r = tf.keras.layers.Dense(128)(r)
    x = tf.keras.layers.Add()([shortcut, r])
    x = tf.keras.layers.Activation('swish')(x)
    x = tf.keras.layers.LayerNormalization()(x)

    # Residual Block 2
    shortcut = x
    r = tf.keras.layers.Dense(128, activation='swish')(x)
    r = tf.keras.layers.Dropout(0.1)(r)
    r = tf.keras.layers.Dense(128)(r)
    x = tf.keras.layers.Add()([shortcut, r])
    x = tf.keras.layers.Activation('swish')(x)

    # Output head + direct linear skip from inputs
    linear_skip = tf.keras.layers.Dense(1)(inputs)
    nonlinear_out = tf.keras.layers.Dense(1)(x)
    outputs = tf.keras.layers.Add()([linear_skip, nonlinear_out])

    return tf.keras.Model(inputs, outputs)

model = build_tabular_resnet(X_train.shape[1])

# Cosine Annealing LR schedule for smooth convergence
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.003,
    decay_steps=200 * (len(X_train) // 16),
    alpha=1e-5
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
    loss='huber',  # Huber loss resists sensor outliers better than pure MSE
    metrics=['mae']
)

model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    verbose=0
)

# 4. Evaluate on Untouched 20% Holdout Test Set
y_pred = model.predict(X_test, verbose=0).flatten()
print(f"Tabular ResNet Holdout MAE  : {mean_absolute_error(y_test, y_pred):.4f} MW")
print(f"Tabular ResNet Holdout RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} MW")
print(f"Tabular ResNet Holdout R2   : {r2_score(y_test, y_pred):.6f}")

Tabular ResNet Holdout MAE  : 0.6316 MW
Tabular ResNet Holdout RMSE : 0.9743 MW
Tabular ResNet Holdout R2   : 0.975812
